In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

# Project Data Dictionary

Generate the shared data dictionary for all source and team-created fields.

Unknown predictor meanings stay marked as `unknown`. No field meaning is inferred from the anonymous names.

## Setup

Use the authoritative source and the Gate 1 profile to build the dictionary.

In [ ]:
def find_repo_root():
    path = Path.cwd().resolve()

    for parent in [path, *path.parents]:
        if (parent / "data" / "allstate_claims_data.csv").exists():
            return parent

    raise FileNotFoundError("Could not find data/allstate_claims_data.csv")


REPO_ROOT = find_repo_root()

DATA_PATH = REPO_ROOT / "data" / "allstate_claims_data.csv"

GATE1_DIR = (
    REPO_ROOT
    / "notebooks"
    / "final-deliverables"
    / "September"
    / "Gate 1"
)

GATE2_DIR = (
    REPO_ROOT
    / "notebooks"
    / "final-deliverables"
    / "September"
    / "Gate 2"
)

PROFILE_PATH = GATE1_DIR / "profile.csv"
DICTIONARY_PATH = GATE2_DIR / "data_dictionary.csv"

DICTIONARY_VERSION = "1.0.0"

ID_COL = "id"
CAT_COLS = [f"cat{i}" for i in range(1, 117)]
CONT_COLS = [f"cont{i}" for i in range(1, 15)]
TARGET_COL = "loss"

EXPECTED_SOURCE_FIELDS = (
    [ID_COL]
    + CAT_COLS
    + CONT_COLS
    + [TARGET_COL]
)

print(f"Source: {DATA_PATH.relative_to(REPO_ROOT)}")
print(f"Profile: {PROFILE_PATH.relative_to(REPO_ROOT)}")
print(f"Output: {DICTIONARY_PATH.relative_to(REPO_ROOT)}")

## Load Source and Profile

The profile provides the observed types, unique counts, ranges, and missingness from the clean Gate 1 workflow.

In [ ]:
df = pd.read_csv(DATA_PATH)
profile = pd.read_csv(PROFILE_PATH)

print(f"Source shape: {df.shape}")
print(f"Profile rows: {len(profile)}")

In [ ]:
expected_profile_fields = set(EXPECTED_SOURCE_FIELDS)
actual_profile_fields = set(profile["column"])

if expected_profile_fields != actual_profile_fields:
    missing = expected_profile_fields - actual_profile_fields
    extra = actual_profile_fields - expected_profile_fields

    raise RuntimeError(
        f"Profile fields do not match the expected schema.\n"
        f"Missing: {sorted(missing)}\n"
        f"Extra: {sorted(extra)}"
    )

print("Profile schema: PASS")

## Source Field Rules

Apply the documented roles and handling rules consistently by field family.

In [ ]:
def source_metadata(field):
    if field == "id":
        return {
            "approved_role": "identifier",
            "unit_or_scale": "identifier",
            "documented_meaning": "unknown",
            "quality_limitation": (
                "Identifier only; no analytical meaning is documented."
            ),
            "september_handling": (
                "Use for integrity checks only. Exclude from distribution "
                "and target-effect interpretation."
            ),
            "meaning_status": "unresolved",
        }

    if field in CAT_COLS:
        return {
            "approved_role": "anonymous nominal categorical predictor",
            "unit_or_scale": "nominal categorical",
            "documented_meaning": "unknown",
            "quality_limitation": (
                "Anonymous predictor; semantic meaning is not documented."
            ),
            "september_handling": (
                "Treat as an unordered nominal categorical predictor. "
                "Do not infer a meaning or ordering."
            ),
            "meaning_status": "unresolved",
        }

    if field in CONT_COLS:
        return {
            "approved_role": "anonymous continuous predictor",
            "unit_or_scale": "scaled between 0 and 1",
            "documented_meaning": "unknown",
            "quality_limitation": (
                "Original unit and transformation are not documented."
            ),
            "september_handling": (
                "Treat as a continuous predictor on the delivered 0-to-1 scale. "
                "Do not infer the original unit or transformation."
            ),
            "meaning_status": "unresolved",
        }

    if field == "loss":
        return {
            "approved_role": "regression target",
            "unit_or_scale": "original loss reporting scale",
            "documented_meaning": "total paid claim amount",
            "quality_limitation": (
                "Delivered data contains positive paid claims only; "
                "do not infer why an individual claim is present."
            ),
            "september_handling": (
                "Retain as the regression target and reporting scale. "
                "Keep derived transformations documented separately."
            ),
            "meaning_status": "documented",
        }

    raise ValueError(f"Unexpected field: {field}")

## Observed Values

Record numeric ranges for numeric fields and exact observed levels for categorical fields.

In [ ]:
def get_category_levels(series):
    levels = sorted(series.dropna().astype(str).unique().tolist())

    # JSON keeps the levels machine-readable inside the CSV cell.
    return json.dumps(levels)

## Source Fields

Build one dictionary row for each of the 132 delivered fields.

In [ ]:
profile_lookup = profile.set_index("column")

source_rows = []

for field in EXPECTED_SOURCE_FIELDS:
    p = profile_lookup.loc[field]
    metadata = source_metadata(field)

    if field in CAT_COLS:
        category_levels = get_category_levels(df[field])
        observed_min = None
        observed_max = None
    else:
        category_levels = None
        observed_min = p["min"]
        observed_max = p["max"]

    source_rows.append({
        "field_name": field,
        "field_origin": "source",
        "source_name": field,
        "approved_role": metadata["approved_role"],
        "observed_dtype": p["source_dtype"],
        "observed_min": observed_min,
        "observed_max": observed_max,
        "category_levels": category_levels,
        "n_unique": int(p["n_unique"]),
        "missing_count": int(p["missing"]),
        "missing_pct": float(p["missing"]) / len(df) * 100,
        "unit_or_scale": metadata["unit_or_scale"],
        "documented_meaning": metadata["documented_meaning"],
        "quality_limitation": metadata["quality_limitation"],
        "september_handling": metadata["september_handling"],
        "meaning_status": metadata["meaning_status"],
        "purpose": None,
        "inputs": None,
        "exact_derivation": None,
        "allowed_values": None,
        "field_version": None,
        "owner": None,
        "dictionary_version": DICTIONARY_VERSION,
    })

source_dictionary = pd.DataFrame(source_rows)

print(f"Source fields: {len(source_dictionary)}")
source_dictionary.head()

## Source Dictionary Check

Make sure all 132 source fields are present once and in the documented order.

In [ ]:
source_dictionary_check = (
    len(source_dictionary) == 132
    and source_dictionary["field_name"].tolist() == EXPECTED_SOURCE_FIELDS
    and source_dictionary["field_name"].is_unique
)

print(f"132 source fields: {len(source_dictionary) == 132}")
print(
    "Correct field order:",
    source_dictionary["field_name"].tolist() == EXPECTED_SOURCE_FIELDS
)
print(
    "Unique field names:",
    source_dictionary["field_name"].is_unique
)

if not source_dictionary_check:
    raise RuntimeError("Source dictionary validation failed.")

## Derived Fields

Team-created fields are documented separately with their purpose, inputs, exact derivation, allowed values, version, and owner.

Only finalized fields should be added here.

In [ ]:
DERIVED_FIELDS = [
    {
        "field_name": "log1p_loss",
        "field_origin": "derived",
        "source_name": None,
        "approved_role": "derived target",
        "observed_dtype": "float64",
        "observed_min": None,
        "observed_max": None,
        "category_levels": None,
        "n_unique": None,
        "missing_count": None,
        "missing_pct": None,
        "unit_or_scale": "log1p transformation of loss",
        "documented_meaning": "log-transformed total paid claim amount",
        "quality_limitation": (
            "Derived scale is not the original reporting or evaluation scale."
        ),
        "september_handling": (
            "Use only where the log-transformed target is explicitly called for. "
            "Keep original loss as the reporting and evaluation scale."
        ),
        "meaning_status": "documented",
        "purpose": (
            "Support target-distribution analysis while reducing the visual "
            "effect of the long upper tail."
        ),
        "inputs": "loss",
        "exact_derivation": "log1p_loss = ln(1 + loss)",
        "allowed_values": "[0, infinity)",
        "field_version": "1.0",
        "owner": "Liam",
        "dictionary_version": DICTIONARY_VERSION,
    },
]

## Calculate Derived Field Profiles

Where possible, calculate observed metadata for finalized derived fields from their documented derivations.

In [ ]:
derived_df = df.copy()

derived_df["log1p_loss"] = np.log1p(
    derived_df["loss"]
)

In [ ]:
derived_rows = []

for metadata in DERIVED_FIELDS:
    field = metadata["field_name"]

    row = metadata.copy()

    if field in derived_df.columns:
        series = derived_df[field]

        row["observed_dtype"] = str(series.dtype)
        row["n_unique"] = int(series.nunique(dropna=False))
        row["missing_count"] = int(series.isna().sum())
        row["missing_pct"] = float(series.isna().mean() * 100)

        if pd.api.types.is_numeric_dtype(series):
            row["observed_min"] = float(series.min())
            row["observed_max"] = float(series.max())

    derived_rows.append(row)

derived_dictionary = pd.DataFrame(derived_rows)

derived_dictionary

## Combined Dictionary

Combine the delivered and team-created fields into one shared dictionary.

In [ ]:
data_dictionary = pd.concat(
    [
        source_dictionary,
        derived_dictionary,
    ],
    ignore_index=True
)

print(f"Source fields: {len(source_dictionary)}")
print(f"Derived fields: {len(derived_dictionary)}")
print(f"Total fields: {len(data_dictionary)}")

data_dictionary

## Dictionary Check

Check that required metadata is present before saving the final artifact.

In [ ]:
required_source_columns = [
    "field_name",
    "source_name",
    "approved_role",
    "observed_dtype",
    "n_unique",
    "missing_count",
    "missing_pct",
    "unit_or_scale",
    "documented_meaning",
    "quality_limitation",
    "september_handling",
]

source_missing_metadata = (
    source_dictionary[required_source_columns]
    .isna()
    .sum()
)

source_missing_metadata

In [ ]:
required_derived_columns = [
    "field_name",
    "approved_role",
    "purpose",
    "inputs",
    "exact_derivation",
    "allowed_values",
    "field_version",
    "owner",
]

derived_missing_metadata = (
    derived_dictionary[required_derived_columns]
    .isna()
    .sum()
)

derived_missing_metadata

In [ ]:
source_metadata_ok = (
    source_missing_metadata.sum() == 0
)

derived_metadata_ok = (
    derived_missing_metadata.sum() == 0
)

unique_names = data_dictionary["field_name"].is_unique

print(f"Source metadata complete: {source_metadata_ok}")
print(f"Derived metadata complete: {derived_metadata_ok}")
print(f"Field names unique: {unique_names}")

if not (
    source_metadata_ok
    and derived_metadata_ok
    and unique_names
):
    raise RuntimeError(
        "Data dictionary is incomplete. "
        "Review the checks above before saving."
    )

## Unknown Meanings

Anonymous predictor meanings should remain `unknown` unless source documentation provides an approved mapping.

In [ ]:
anonymous_fields = source_dictionary[
    source_dictionary["field_name"].isin(
        CAT_COLS + CONT_COLS
    )
]

incorrect_meanings = anonymous_fields[
    anonymous_fields["documented_meaning"] != "unknown"
]

print(
    f"Anonymous predictors marked unknown: "
    f"{len(anonymous_fields) - len(incorrect_meanings)}/"
    f"{len(anonymous_fields)}"
)

incorrect_meanings

## Save Data Dictionary

Save the validated dictionary to the shared September deliverables.

In [ ]:
data_dictionary.to_csv(
    DICTIONARY_PATH,
    index=False
)

print(
    f"Saved {len(data_dictionary)} fields to "
    f"{DICTIONARY_PATH.relative_to(REPO_ROOT)}"
)

## Result

In [ ]:
print("=" * 55)
print("DATA DICTIONARY")
print("=" * 55)
print(f"Dictionary version: {DICTIONARY_VERSION}")
print(f"Source fields:      {len(source_dictionary)}")
print(f"Derived fields:     {len(derived_dictionary)}")
print(f"Total fields:       {len(data_dictionary)}")
print(f"Unknown cat/cont:   {len(anonymous_fields)}")
print("=" * 55)
print("DATA DICTIONARY: PASS")